# ADV Processing

Utilizes the Dolfyn package. Documentation found at https://dolfyn.readthedocs.io/en/latest/index.html

- attaches metadata to each sensor
- groups all sensor files for each sensor
- converts .vec files to .nc (xarrays)

Each sensor has 4 `.vec` files. The **last** file of each group is cut off mid-ensemble
(the instrument was still writing when it was recovered), so dolfyn raises on it and
returns nothing. `safe_read` below walks the file to find the exact last good ensemble
and re-reads up to that point.

Run order: setup -> metadata -> helpers -> (optional single-sensor test) -> batch.

To do:
- ~~make script to just truncate the last ensemble of each of the last files for each sensor~~
- kvh .txt files for recovery and for deployment, log those on the .csv
- inspect what kind of metadata .vec files already have
- have clean metadata file

In [2]:
import os

import dolfyn
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path
from dolfyn.io import nortek

##-----setup-----##

# pathways
data_dir = Path("../data/raw/vectors")
out_dir = Path("../data/processed")   # under data/, which is gitignored
out_dir.mkdir(parents=True, exist_ok=True)

# are we in the right place? are there files?
vec_files = sorted(data_dir.glob("*.vec")) + sorted(data_dir.glob("*.VEC"))
print(f"Found {len(vec_files)} .vec files")

Found 32 .vec files


In [3]:
##-----metadata handling-----##

# sensor_notes.csv is written for humans: spreadsheet-style headers, '"' as a
# ditto mark, and a few blank rows at the bottom. Map the columns we care about
# onto names worth putting in a netCDF header and sanitize whatever is left.
COLUMN_RENAMES = {
    # current sensor_notes.csv headers (2026-09-28)
    "Processed Data Sensor Name (.nc file)": "sensor_id",
    "Deployment Latitude (WGS84)": "latitude",
    "Deployment Longitude (WGS84)": "longitude",
    "S/N": "serial_number",
    # older headers, kept so an older copy of the CSV still reads
    "lat": "latitude",
    "lon": "longitude",
    "Deployment Depth MSL(m), approximate": "instrument_depth_m",
    "Magnetic Heading (KVH)": "heading_kvh_magnetic_deg",   # magnetic, not true north
    "Shoreline Heading W to E (deg true)": "shoreline_heading_deg_true",   # per transect, Google Earth
    "Shore Normal Onshore (deg true)": "shore_normal_onshore_deg_true",   # = shoreline - 90
    "Clock Drift (seconds) over Deployment Duration***": "clock_drift_sec",
    "Backed up to HNL Google Drive?": "backed_up_to_gdrive",
}

FLOAT_ATTRS = ["latitude", "longitude", "instrument_depth_m",
               "heading_kvh_magnetic_deg", "clock_drift_sec",
               "shoreline_heading_deg_true", "shore_normal_onshore_deg_true"]


def _sanitize(name):
    """Spreadsheet header -> snake_case attribute name."""
    out = "".join(c if (c.isalnum() or c == "_") else "_" for c in name.strip().lower())
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


sensor_meta = pd.read_csv("../data/metadata/sensor_notes.csv", dtype=str, skipinitialspace=True)
sensor_meta.columns = sensor_meta.columns.str.strip()
sensor_meta = sensor_meta.drop(
    columns=[c for c in sensor_meta.columns if c.startswith("Unnamed:")])
sensor_meta = sensor_meta.rename(columns=COLUMN_RENAMES)
sensor_meta.columns = [_sanitize(c) for c in sensor_meta.columns]

# drop the blank trailing rows
sensor_meta["sensor_id"] = sensor_meta["sensor_id"].fillna("").str.strip()
sensor_meta = sensor_meta[sensor_meta["sensor_id"] != ""].set_index("sensor_id")

manifest = pd.read_csv("../data/metadata/manifest.csv", dtype=str, skipinitialspace=True)
manifest.columns = manifest.columns.str.strip()
manifest["filename"] = manifest["filename"].str.strip()
manifest["sensor_id"] = manifest["sensor_id"].str.strip()

sensor_groups = manifest.groupby("sensor_id")["filename"].apply(list).to_dict()

print("Sensor IDs found in manifest:", list(sensor_groups.keys()))
print("Sensor IDs found in metadata:", list(sensor_meta.index))

# cross-check the manifest against what is actually on disk
listed = {f for fns in sensor_groups.values() for f in fns}
on_disk = {p.name for p in vec_files}
if listed - on_disk:
    print("WARNING: manifest lists files not on disk:", sorted(listed - on_disk))
if on_disk - listed:
    print("WARNING: files on disk not in manifest:", sorted(on_disk - listed))

no_files = [s for s in sensor_meta.index
            if s.startswith("V") and s not in sensor_groups]
if no_files:
    print("NOTE: vectors in sensor_notes.csv with no files in the manifest:", no_files)

Sensor IDs found in manifest: ['VA1', 'VB1', 'VC1', 'VC2', 'VD1', 'VD2', 'VE1', 'VE2']
Sensor IDs found in metadata: ['VA1', 'VB1', 'VD2', 'VC1', 'VC2', 'VE2', 'VD1', 'VE3', 'VE1', 'ADCP', 'PB1', 'PB2', 'PC1', 'PE1', 'PD1']
NOTE: vectors in sensor_notes.csv with no files in the manifest: ['VE3']


In [5]:
##-----end-of-file truncation-----##

# The last file in each sensor group is cut off mid-ensemble: the instrument was
# still writing when it was pulled, so the final record carries a garbage
# timestamp and dolfyn raises `ValueError: month must be in 1..12` -- returning
# nothing for an otherwise complete 100 MB file.
#
# To recover it we walk the file once to find the exact ensemble where parsing
# stops, then re-read up to that point with `nens=`.
#
# Deliberately NOT estimated from file size: measured against HI-0202_4.VEC, a
# bytes-per-ensemble estimate taken from a clean file came out 1.27% high
# (2,755,545 vs the 2,720,606 that actually parse), because the mix of sysdata
# and config records differs between files. Guessing from that would silently
# discard ~1.4 h of good record off the end of every sensor. The walk is exact.

# How close to EOF a failure has to be before we call it "the instrument was
# still writing" rather than real corruption in the middle of the record.
EOF_TOLERANCE_BYTES = 64 * 1024


def probe_last_good_ensemble(fpath):
    """Walk a .vec file and report where parsing stops.

    Returns (last_good_ensemble, fail_byte, file_size, error). `last_good_ensemble`
    is None if the file parses all the way through. This pass is cheap (~14 s/file
    vs ~80 s for a real read) because it never builds the xarray Dataset.
    """
    size = os.path.getsize(fpath)
    state = {"c": 0, "pos": 0}
    original = nortek._NortekReader.readnext

    def traced(self):
        state["c"] = self.c
        state["pos"] = self.f.tell()
        return original(self)

    nortek._NortekReader.readnext = traced
    try:
        dolfyn.read(fpath, userdata=True)
        return None, None, size, None
    except Exception as e:
        return state["c"], state["pos"], size, e
    finally:
        nortek._NortekReader.readnext = original


def safe_read(fn, data_dir, log):
    """Read a .vec file, recovering from end-of-file truncation.

    Records any truncation in `log` (a list of dicts). Raises if the file fails
    anywhere other than the end -- that is real corruption, not a dirty stop, and
    should not be silently truncated away.
    """
    fpath = str(data_dir / fn)

    try:
        return dolfyn.read(fpath, userdata=True)
    except Exception as first_error:
        print(f"  {fn}: full read failed ({first_error})"
              f" -- locating last good ensemble...")

    last_good, fail_byte, size, probe_error = probe_last_good_ensemble(fpath)

    if last_good is None:
        raise RuntimeError(
            f"{fn}: read failed but the file parses end to end, so the problem is "
            f"in Dataset construction rather than truncation. Original error: "
            f"{first_error}")

    bytes_from_eof = size - fail_byte
    if bytes_from_eof > EOF_TOLERANCE_BYTES:
        raise RuntimeError(
            f"{fn}: parsing stopped at byte {fail_byte:,} of {size:,} "
            f"({100 * fail_byte / size:.2f}% through, {bytes_from_eof:,} bytes from "
            f"EOF). That is mid-file corruption, not a dirty instrument stop, so it "
            f"is not being truncated. Underlying error: {probe_error}")

    ds = dolfyn.read(fpath, userdata=True, nens=last_good)
    n = ds.sizes["time"]
    print(f"  {fn}: recovered {n:,} ensembles through {ds.time.values[-1]} "
          f"(parse stopped {bytes_from_eof:,} bytes from EOF)")

    log.append({
        "filename": fn,
        "recovered_ensembles": int(n),
        "last_good_time": str(ds.time.values[-1]),
        "fail_byte": int(fail_byte),
        "file_size_bytes": int(size),
        "bytes_from_eof": int(bytes_from_eof),
    })
    return ds

In [6]:
##-----per-sensor load and metadata-----##

def load_sensor(sensor_id, filenames, data_dir, truncation_log):
    """Read and concatenate all four files for one sensor.

    Consecutive files can share a look-back overlap at their boundary: on
    VA1 this was ~4.9 h between files 1-2 and ~10 s between files 2-3 and
    3-4, all four boundaries overlapping. This looks like the recorder/
    download software re-including a buffered tail in the next file so nothing
    is lost across a file split.

    The overlapping samples are NOT exact timestamp duplicates -- checked on
    VA1, where the "same" sample recorded at the tail of one file and the head
    of the next differed by ~12 microseconds of encoding jitter (e.g.
    ...999983787 vs ...999996185). A dedup keyed on exact time equality
    (`np.unique` on the time array) therefore keeps both copies: verified
    that VA1 processed with dedup saved exactly the sum of the 4 raw files'
    sample counts, i.e. 0 duplicates removed, leaving ~4.9 h of doubled
    sample density in the merged record.

    Fix: trim each file's tail to strictly before the next file's first
    timestamp, using the boundary time rather than matching individual
    values. Cheap (one boolean mask per boundary) and exact regardless of
    overlap size.
    """
    filenames_sorted = sorted(filenames)
    datasets = []
    for fn in filenames_sorted:
        ds = safe_read(fn, data_dir, truncation_log)
        print(f"  {fn}: {ds.sizes['time']:>10,} samples   "
              f"{ds.time.values[0]} -> {ds.time.values[-1]}")
        datasets.append(ds)

    for i in range(len(datasets) - 1):
        last_time = datasets[i].time.values[-1]
        next_start = datasets[i + 1].time.values[0]
        gap_s = (next_start - last_time) / np.timedelta64(1, "s")

        if last_time >= next_start:
            keep = datasets[i].time.values < next_start
            n_drop = int((~keep).sum())
            print(f"  {filenames_sorted[i]}: trimming {n_drop:,} samples "
                  f"({-gap_s:.1f}s) that overlap {filenames_sorted[i+1]}")
            datasets[i] = datasets[i].isel(time=keep)
        elif gap_s > 5:
            print(f"  NOTE: {gap_s:.1f}s gap between {filenames_sorted[i]} "
                  f"and {filenames_sorted[i+1]}")

    # data_vars/coords="minimal" keeps static variables (beam2inst_orientmat)
    # from being broadcast along time -- on its own that was ~800 MB per sensor.
    merged = xr.concat(datasets, dim="time", data_vars="minimal",
                       coords="minimal", compat="override", join="exact")
    del datasets

    t = merged.time.values
    if not (t[1:] > t[:-1]).all():
        n_bad = int((t[1:] <= t[:-1]).sum())
        raise RuntimeError(
            f"{sensor_id}: {n_bad} non-increasing time step(s) remain after "
            f"trimming known file-boundary overlaps -- something other than a "
            f"simple boundary overlap is going on here. Investigate before "
            f"trusting this sensor's output.")

    return merged


def apply_metadata(ds, sensor_id, meta_df, sensor_truncations):
    """Attach the sensor_notes.csv row for this sensor as netCDF attributes."""
    row = meta_df.loc[sensor_id]

    attrs = {}
    for key, val in row.items():
        if val is None or (isinstance(val, float) and np.isnan(val)):
            continue
        val = str(val).strip()
        # '"' is the spreadsheet ditto mark for "same as the row above"
        if val in ("", "nan", '"', '""', '""""'):
            continue
        attrs[key] = val

    for key in FLOAT_ATTRS:
        if key in attrs:
            try:
                attrs[key] = float(attrs[key])
            except ValueError:
                print(f"WARNING: {sensor_id}: could not read {key}={attrs[key]!r} as a number")

    # Never let deployment notes overwrite dolfyn's own attrs -- `orientation`
    # and `sample_rate` sit right next to `coord_sys` and `fs`, which the
    # rotation code downstream depends on.
    collisions = sorted(set(attrs) & set(ds.attrs))
    if collisions:
        print(f"  prefixing deployment_ onto {collisions} to protect dolfyn attrs")
        for key in collisions:
            attrs[f"deployment_{key}"] = attrs.pop(key)

    ds.attrs.update(attrs)
    ds.attrs["sensor_id"] = sensor_id

    if sensor_truncations:
        notes = "; ".join(
            f"{t['filename']}: truncated at EOF, kept {t['recovered_ensembles']:,} "
            f"ensembles through {t['last_good_time']} (parse stopped "
            f"{t['bytes_from_eof']} bytes from EOF)"
            for t in sensor_truncations
        )
        ds.attrs["processing_notes"] = (
            ds.attrs.get("processing_notes", "") + " " + notes
        ).strip()

    required = ["latitude", "longitude", "instrument_depth_m"]
    missing = [k for k in required if k not in ds.attrs or ds.attrs[k] in (None, "")]
    if missing:
        print(f"WARNING: missing metadata for {sensor_id}: {missing}")

    return ds

In [ ]:
##-----single sensor test-----##
# Optional: run one sensor end to end before kicking off the batch below.
# Takes about 5 minutes. Skip straight to the batch cell if you don't need it.

sensor_id = list(sensor_groups.keys())[0]
truncation_log = []

ds = load_sensor(sensor_id, sensor_groups[sensor_id], data_dir, truncation_log)
ds = apply_metadata(ds, sensor_id, sensor_meta, truncation_log)
ds

In [ ]:
# quick look at the compass heading.
# numpy mean rather than .values.tolist() -- 11M samples as a Python list is
# ~500 MB and this machine has 8 GB.
print(f"mean heading, samples 10k-100k: {np.nanmean(ds['heading'].values[10000:100000]):7.2f} deg")
print(f"mean heading, whole record:     {np.nanmean(ds['heading'].values):7.2f} deg")

In [ ]:
##-----batch: all sensors-----##
# ~5-7 min per sensor. Each .nc is written inside the loop, so this is safe to
# interrupt -- re-running skips sensors that already have output.

all_truncations = {}   # sensor_id -> list of truncation records
skipped = []

for sensor_id, filenames in sensor_groups.items():
    nc_path = out_dir / f"{sensor_id}.nc"
    if nc_path.exists():
        print(f"{sensor_id}: {nc_path.name} already exists, skipping")
        continue

    if sensor_id not in sensor_meta.index:
        print(f"SKIPPING {sensor_id}: not in sensor_notes.csv")
        skipped.append(sensor_id)
        continue

    print(f"\nProcessing {sensor_id} ({len(filenames)} files)...")
    truncation_log = []

    ds = load_sensor(sensor_id, filenames, data_dir, truncation_log)
    ds = apply_metadata(ds, sensor_id, sensor_meta, truncation_log)

    if truncation_log:
        all_truncations[sensor_id] = truncation_log

    # compression=True is worth it here: ~94 -> ~33 bytes/sample, and the disk
    # this is running on is nearly full.
    dolfyn.save(ds, str(nc_path), compression=True)
    print(f"{sensor_id}: saved {ds.sizes['time']:,} samples -> {nc_path.name} "
          f"({nc_path.stat().st_size / 1e9:.2f} GB)")

    del ds

if skipped:
    print(f"\n{len(skipped)} sensor(s) skipped for missing metadata: {skipped}")

if all_truncations:
    print("\n--- Truncation summary ---")
    for sid, entries in all_truncations.items():
        for e in entries:
            print(f"  {sid}/{e['filename']}: kept {e['recovered_ensembles']:,} "
                  f"ensembles through {e['last_good_time']}")

In [ ]:
##-----slim down variables-----##
# dolfyn keeps every housekeeping channel the Vector records. Writes a slimmed
# copy of each sensor to data/processed/slim/ and leaves the full files alone.
#
# Analysis variables, pulled out under plain names:
#   p          pressure (dbar)
#   u, v, w    velocity X, Y, Z in the INSTRUMENT frame -- not east/north yet.
#              w is kept because rotating to east/north mixes all three.
#
# Kept until despiking + rotation are done:
#   corr, amp            QC / despiking thresholds (uint8, nearly free)
#   heading/pitch/roll   internal compass & tilt (compass suspect -- see
#                        figs/orientation; KVH heading is in attrs)
#   orientation_down, beam2inst_orientmat, temp
#
# Dropped (summarized into attrs): orientmat (rebuilt from h/p/r), batt,
# c_sound, error, status.
#
# dolfyn's tools (rotate2, despiking) expect a `vel` (dir, time) variable. To
# hand one of these files back to dolfyn, rebuild it with to_dolfyn() below.

slim_dir = out_dir / "slim"
slim_dir.mkdir(exist_ok=True)

DROP_VARS = ["orientmat", "batt", "c_sound", "error", "status"]


def _counts(da):
    u, c = np.unique(da.values, return_counts=True)
    return "; ".join(f"{int(k)}: {int(n)}" for k, n in zip(u, c))


for nc_path in sorted(out_dir.glob("*.nc")):
    slim_path = slim_dir / nc_path.name
    if slim_path.exists():
        print(f"{nc_path.stem}: slim file already exists, skipping")
        continue

    ds = dolfyn.load(str(nc_path))

    od = ds["orientation_down"].values
    if not (od.all() or (~od).all()):
        print(f"  WARNING {nc_path.stem}: orientation_down changes during the record")

    summary = {
        "batt_min_V": float(ds["batt"].min()),
        "batt_max_V": float(ds["batt"].max()),
        "c_sound_min_m_s": float(ds["c_sound"].min()),
        "c_sound_max_m_s": float(ds["c_sound"].max()),
        "error_code_counts": _counts(ds["error"]),
        "status_code_counts": _counts(ds["status"]),
        "orientation_down_constant": str(bool(od.all()) if (od.all() or (~od).all()) else "varies"),
    }

    slim = ds.drop_vars(DROP_VARS).load()
    ds.close()
    vel = slim["vel"]
    for comp, d in zip(["u", "v", "w"], ["X", "Y", "Z"]):
        slim[comp] = vel.sel(dir=d, drop=True)
        slim[comp].attrs = {**vel.attrs,
                            "long_name": f"Water velocity, instrument {d}",
                            "ref_frame": "inst"}
    slim = slim.drop_vars(["vel", "dir"]).rename({"pressure": "p"})
    slim.attrs.update(summary)
    slim.attrs["dropped_variables"] = ", ".join(DROP_VARS)
    # On reload, the string coords (dir/earth/inst) are variable-length strings,
    # which netCDF refuses to zlib ("Filter error ... variable 'dir'"). Compress
    # everything else. The full encoding dict is passed explicitly because
    # dolfyn.save drops its own compression settings whenever `encoding=` is given.
    enc = {}
    for name, v in slim.variables.items():
        v.encoding = {}
        enc[name] = ({"zlib": False} if v.dtype.kind in "OU"
                     else {"zlib": True, "complevel": 1})

    dolfyn.save(slim, str(slim_path), encoding=enc)
    print(f"{nc_path.stem}: {nc_path.stat().st_size / 1e6:6.0f} MB -> "
          f"{slim_path.stat().st_size / 1e6:6.0f} MB   "
          f"errors [{summary['error_code_counts']}]  "
          f"orientation_down={summary['orientation_down_constant']}")
    del slim


def to_dolfyn(ds):
    """Re-pack u/v/w and p into the vel/pressure layout dolfyn's tools expect."""
    out = ds.rename({"p": "pressure"})
    out["vel"] = xr.concat([ds["u"], ds["v"], ds["w"]],
                           dim=pd.Index(["X", "Y", "Z"], name="dir"))
    out["vel"].attrs = {**ds["u"].attrs, "long_name": "Water Velocity"}
    out.attrs["rotate_vars"] = ["vel"]
    return out.drop_vars(["u", "v", "w"])


In [8]:
##-----refresh deployment metadata in existing files-----##
# Re-applies sensor_notes.csv to the .nc files already in data/processed/
# without re-reading the .vec files: attributes are rewritten in place with
# netCDF4, so no 300 MB rewrite per file. Run the metadata cell first.
import netCDF4

# attrs from older versions of sensor_notes.csv that have since been renamed
RETIRED_ATTRS = ["heading_kvh_deg"]


def refresh_metadata(nc_path, sensor_id):
    with netCDF4.Dataset(nc_path, "a") as nc:
        old = {k: nc.getncattr(k) for k in nc.ncattrs()}

        # Strip what the previous pass took from sensor_notes.csv, so
        # apply_metadata doesn't mistake it for dolfyn's own attrs and prefix
        # duplicates. `x` stays when `deployment_x` exists: that means x
        # belongs to dolfyn (e.g. serial_number = 'VEC 8190').
        base = dict(old)
        for col in set(sensor_meta.columns) | set(RETIRED_ATTRS):
            if f"deployment_{col}" in base:
                del base[f"deployment_{col}"]
            elif col in base:
                del base[col]

        # same apply_metadata the batch uses, on an empty stand-in, so renames,
        # ditto marks and dolfyn-attr collisions are handled identically
        new = apply_metadata(xr.Dataset(attrs=base), sensor_id,
                             sensor_meta, []).attrs

        changed = {k: v for k, v in new.items()
                   if k not in old or str(old[k]) != str(v)}
        removed = [k for k in old if k not in new]
        for k, v in changed.items():
            nc.setncattr(k, v)
        for k in removed:
            nc.delncattr(k)
    return changed, removed


for nc_path in sorted(out_dir.glob("*.nc")):
    sid = nc_path.stem
    try:
        changed, removed = refresh_metadata(str(nc_path), sid)
    except OSError as e:
        # HDF5 won't open a file for writing while another kernel has it open
        print(f"{nc_path.name:12s} SKIPPED, file is locked (open in another "
              f"notebook? restart that kernel and re-run): {e}")
        continue
    where = nc_path.relative_to(out_dir)
    if changed or removed:
        print(f"{str(where):12s} updated {sorted(changed)}"
              + (f"  removed {removed}" if removed else ""))
    else:
        print(f"{str(where):12s} already current")


  prefixing deployment_ onto ['serial_number'] to protect dolfyn attrs
VA1.nc       updated ['heading_kvh_magnetic_deg']  removed ['heading_kvh_deg']
  prefixing deployment_ onto ['serial_number'] to protect dolfyn attrs
VB1.nc       already current
  prefixing deployment_ onto ['serial_number'] to protect dolfyn attrs
VC1.nc       already current
  prefixing deployment_ onto ['serial_number'] to protect dolfyn attrs
VC2.nc       already current
  prefixing deployment_ onto ['serial_number'] to protect dolfyn attrs
VD1.nc       already current
  prefixing deployment_ onto ['serial_number'] to protect dolfyn attrs
VD2.nc       already current
  prefixing deployment_ onto ['serial_number'] to protect dolfyn attrs
VE1.nc       already current
  prefixing deployment_ onto ['serial_number'] to protect dolfyn attrs
VE2.nc       already current


In [9]:
##-----quick look: internal compass/tilt vs KVH-----##
# The internal compass is suspect (steel frame), so before rotating anything,
# look at heading/pitch/roll through each deployment against the KVH heading
# taken at install (sensor_notes.csv). 1-min circular means: heading wraps at
# 0/360 and pitch/roll sit near +-180, so a plain mean would be garbage.
# Gray bands = pressure < 3 dbar (out of the water;
# in-air reads ~1.2-1.5 dbar, shallowest sensors are ~4 m).
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig_dir = Path("../figs/orientation")
fig_dir.mkdir(parents=True, exist_ok=True)

INK, INK2, GRID, SERIES = "#0b0b0b", "#52514e", "#e4e3df", "#2a78d6"
plt.rcParams.update({
    "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2,
    "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.facecolor": "white", "font.size": 9,
})


def circmean_1min(da, lo):
    """1-min circular mean of an angle in degrees, wrapped to [lo, lo+360)."""
    r = np.radians(da.values)
    df = pd.DataFrame({"s": np.sin(r), "c": np.cos(r)},
                      index=pd.DatetimeIndex(da.time.values)).resample("1min").mean()
    return (np.degrees(np.arctan2(df.s, df.c)) - lo) % 360 + lo


def dry_spans(p_1min, thresh=3.0):
    dry = (p_1min < thresh).values
    t = p_1min.index
    spans, start = [], None
    for i, d in enumerate(dry):
        if d and start is None:
            start = t[i]
        elif not d and start is not None:
            spans.append((start, t[i])); start = None
    if start is not None:
        spans.append((start, t[-1]))
    return spans


summary = {}
for nc_path in sorted(out_dir.glob("*.nc")):
    sid = nc_path.stem
    ds = xr.open_dataset(nc_path)
    hdg = circmean_1min(ds["heading"], 0)
    pit = circmean_1min(ds["pitch"], -180)
    rol = circmean_1min(ds["roll"], -180)
    prs = ds["p"].to_series().resample("1min").median()
    kvh = ds.attrs.get("heading_kvh_magnetic_deg")
    ds.close()
    summary[sid] = (hdg, kvh, dry_spans(prs))

    fig, axs = plt.subplots(4, 1, figsize=(10, 8), sharex=True,
                            gridspec_kw={"height_ratios": [1.4, 1, 1, 0.8]})
    for ax, s, lab, lim in [(axs[0], hdg, "heading (deg)", (0, 360)),
                            (axs[1], pit, "pitch (deg)", (-180, 180)),
                            (axs[2], rol, "roll (deg)", (-180, 180)),
                            (axs[3], prs, "pressure (dbar)", None)]:
        ax.plot(s.index, s.values, ".", ms=1.2, color=SERIES, rasterized=True)
        ax.set_ylabel(lab)
        if lim:
            ax.set_ylim(*lim); ax.set_yticks(np.linspace(*lim, 5))
        for a, b in summary[sid][2]:
            ax.axvspan(a, b, color=GRID, alpha=0.7, lw=0)
    if kvh is not None:
        axs[0].axhline(kvh, color=INK, lw=1.2, ls="--")
        axs[0].text(axs[0].get_xlim()[1], kvh, f" KVH {kvh:.1f}° mag", va="center",
                    color=INK, fontsize=8, clip_on=False)
    else:
        axs[0].text(0.01, 0.95, "no KVH heading in sensor_notes.csv",
                    transform=axs[0].transAxes, va="top", color=INK2, fontsize=8)
    note = ds.attrs.get("notes", "")
    axs[0].set_title(f"{sid}  internal compass & tilt, 1-min circular means"
                     + (f"   [{note}]" if note else ""), loc="left", color=INK)
    axs[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.tight_layout()
    fig.savefig(fig_dir / f"{sid}_orientation.png", dpi=150)
    plt.close(fig)
    print(f"{sid}: median heading {np.nanmedian(hdg):6.1f}  KVH {kvh}  "
          f"median pitch {np.nanmedian(pit):7.1f}  roll {np.nanmedian(rol):7.1f}")

# overview: heading only, all sensors on the same 0-360 scale
fig, axs = plt.subplots(4, 2, figsize=(11, 9), sharey=True)
for ax, (sid, (hdg, kvh, spans)) in zip(axs.flat, summary.items()):
    ax.plot(hdg.index, hdg.values, ".", ms=1, color=SERIES, rasterized=True)
    for a, b in spans:
        ax.axvspan(a, b, color=GRID, alpha=0.7, lw=0)
    if kvh is not None:
        ax.axhline(kvh, color=INK, lw=1.2, ls="--")
    ax.set_title(sid + ("" if kvh is not None else "  (no KVH)"), loc="left",
                 color=INK, fontsize=9)
    ax.set_ylim(0, 360); ax.set_yticks([0, 90, 180, 270, 360])
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
for ax in axs[:, 0]:
    ax.set_ylabel("heading (deg)")
fig.suptitle("Internal compass heading (1-min circular mean); dashed = KVH (magnetic) at install",
             x=0.01, ha="left", color=INK)
fig.tight_layout()
fig.savefig(fig_dir / "all_sensors_heading.png", dpi=150)
plt.close(fig)


VA1: median heading  190.9  KVH 343.9  median pitch  -156.7  roll  -156.8
VB1: median heading  206.5  KVH 358.8  median pitch  -165.1  roll  -167.2
VC1: median heading  257.6  KVH 341.8  median pitch    19.0  roll  -166.4
VC2: median heading   32.5  KVH 343.2  median pitch   158.0  roll   166.0
VD1: median heading  280.7  KVH 345.0  median pitch   168.2  roll  -173.0
VD2: median heading  316.8  KVH 342.5  median pitch   169.9  roll    22.4
VE1: median heading  302.2  KVH 353.2  median pitch   -28.8  roll    37.0
VE2: median heading  228.7  KVH 345.0  median pitch   -26.4  roll    17.8


In [ ]:
## despiking

In [ ]:
# clock drift offset

In [ ]:
# rotate to heading